# Step 1: CLAP on cat meows (zero-shot vs embeddings + logistic regression)

Behaviour model spike for MyPet (FR02, FR20). Runs free on Google Colab (CPU is enough, a GPU is faster).

**Question:** does a small classifier trained on CLAP audio embeddings beat CLAP zero-shot at telling *why* a cat meows?

- Model: `laion/clap-htsat-unfused` (PyTorch). The app's planned ONNX file `Xenova/clap-htsat-unfused` is a conversion of this same model; checking the ONNX matches is Step 2.
- Data: CatMeows (Ludovico et al., 2020, Zenodo 4008297; check the licence on the record before reuse): 440 meows from 21 cats in 3 contexts: **F** waiting for food, **B** being brushed, **I** isolated in an unfamiliar place.
- Split by cat (no cat in two splits), 70/15/15 as in the proposal. With only 21 cats the test split has about 3 cats, so a grouped 7-fold cross-validation by cat is also reported as the steadier number.
- Dogs (EmotionalCanines) and YAMNet are later steps.

In [ ]:
import os, glob, json, zipfile, urllib.request, time
import numpy as np, librosa, torch
from transformers import ClapModel, ClapProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
SEED = 42
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Download CatMeows

In [ ]:
if not os.path.isdir("catmeows"):
    rec = json.load(urllib.request.urlopen("https://zenodo.org/api/records/4008297"))
    for f in rec["files"]:
        print(f["key"], f["size"])
    zf = [f for f in rec["files"] if f["key"].endswith(".zip")][0]
    urllib.request.urlretrieve(zf["links"]["self"], "catmeows.zip")
    zipfile.ZipFile("catmeows.zip").extractall("catmeows")
files = sorted(glob.glob("catmeows/**/*.wav", recursive=True))
print(len(files), "clips; example:", os.path.basename(files[0]))

In [ ]:
# File name: C_NNNNN_BB_SS_OOOOO_RXX.wav  (C = context, NNNNN = cat id)
CTX = {"F": "waiting for food", "B": "being brushed", "I": "isolated"}
labels = np.array([os.path.basename(f).split("_")[0] for f in files])
cats   = np.array([os.path.basename(f).split("_")[1] for f in files])
classes = sorted(CTX)
y = np.array([classes.index(l) for l in labels])
print("clips per context:", {c: int((labels == c).sum()) for c in classes})
print("cats:", len(set(cats)))

## 2. Load audio the way the app will (48 kHz mono, 10 s)

In [ ]:
SR = 48000
audio = [librosa.load(f, sr=SR, mono=True)[0] for f in files]
d = np.array([len(a) / SR for a in audio])
print(f"clip length: median {np.median(d):.2f} s, min {d.min():.2f} s, max {d.max():.2f} s")

## 3. CLAP embeddings and zero-shot

In [ ]:
model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

def embed_audio(batch):
    # repeatpad short clips to 10 s, as planned for the app
    inp = proc.feature_extractor(batch, sampling_rate=SR, padding="repeatpad", return_tensors="pt")
    with torch.no_grad():
        e = model.get_audio_features(input_features=inp["input_features"].to(device))
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

t0 = time.time()
X = np.concatenate([embed_audio(audio[i:i+16]) for i in range(0, len(audio), 16)])
print("embeddings:", X.shape, f"in {time.time()-t0:.0f} s")

In [ ]:
PROMPTS = {
    "F": "the sound of a hungry cat meowing for food",
    "B": "the sound of a relaxed cat meowing while being brushed",
    "I": "the sound of a stressed cat meowing alone in an unfamiliar place",
}
tok = proc.tokenizer([PROMPTS[c] for c in classes], padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    T = torch.nn.functional.normalize(model.get_text_features(**tok), dim=-1).cpu().numpy()
zs_pred = (X @ T.T).argmax(1)

## 4. Split by cat and score

In [ ]:
def report(name, yt, yp):
    r = {"accuracy": round(accuracy_score(yt, yp), 3),
         "macro_f1": round(f1_score(yt, yp, average="macro"), 3)}
    print(f"{name:42s} acc {r['accuracy']:.3f}  macro F1 {r['macro_f1']:.3f}")
    return r

# 70/15/15 by cat
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr, rest = next(gss.split(X, y, cats))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_i, te_i = next(gss2.split(X[rest], y[rest], cats[rest]))
va, te = rest[va_i], rest[te_i]
assert not (set(cats[tr]) & set(cats[te])) and not (set(cats[va]) & set(cats[te]))
print(f"cats train/val/test: {len(set(cats[tr]))}/{len(set(cats[va]))}/{len(set(cats[te]))}; clips {len(tr)}/{len(va)}/{len(te)}")

# pick C on validation cats only
best = max([0.01, 0.1, 1, 10], key=lambda C: f1_score(
    y[va], make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr], y[tr]).predict(X[va]), average="macro"))
clf = make_pipeline(StandardScaler(), LogisticRegression(C=best, max_iter=5000)).fit(X[tr], y[tr])
print("chosen C:", best)

results = {}
results["majority_test"] = report("Majority class (test cats)", y[te], np.full(len(te), np.bincount(y[tr]).argmax()))
results["zero_shot_test"] = report("CLAP zero-shot (test cats)", y[te], zs_pred[te])
results["logreg_test"] = report("CLAP + logistic regression (test cats)", y[te], clf.predict(X[te]))

# steadier: 7-fold CV by cat (3 cats held out each fold)
cv_pred = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(C=best, max_iter=5000)), X, y, groups=cats, cv=GroupKFold(n_splits=7))
results["zero_shot_all"] = report("CLAP zero-shot (all 440 clips)", y, zs_pred)
results["logreg_cv7"] = report("CLAP + logistic regression (7-fold by cat)", y, cv_pred)
print("\nconfusion matrix, 7-fold by cat (rows true, cols predicted; order", classes, ")")
print(confusion_matrix(y, cv_pred))
print("\nconfusion matrix, zero-shot")
print(confusion_matrix(y, zs_pred))

## 5. Speed on this machine (one 10 s clip)

In [ ]:
one = audio[0]
t0 = time.time()
for _ in range(5): embed_audio([one])
results["seconds_per_clip_colab_" + device] = round((time.time() - t0) / 5, 3)
print(results["seconds_per_clip_colab_" + device], "s per clip on", device, "(a phone will differ; timed in a later step)")
print("\nRESULTS_JSON", json.dumps(results))